# v301 日内／跨日分层 GRU 平台提交

上传本 Notebook 和同目录的 `v301-model.json`，共两个 UTF-8 文本文件。
运行 `v301-global-fp16-s42`，按验证集选定第 7 轮权重。本地判定：**达到本地改善条件**。导出理由：本地验证主指标提高且覆盖未降低。
JSON 包含完整数值权重、三频统计量、配置、依赖及训练／推理源码。FP32 权重以九位有效数字保存，加载后按原 dtype 恢复，并已校验逐张量字节完全一致；统计量保留原精度。
入口为 `main(datasources, start_date, end_date)`。`datasources` 提供 `bar1m`、`bar5m`、`bar30m` 的实际行情表名。导入不查询行情；推理仅使用授权股票池和三频行情，不读取标签或训练。日期字符串的结束日包含全天，时间戳的结束分钟也包含在内。
训练目标为完整截面有限标签的 z-score（总体标准差，`ddof=0`），使用 Huber `delta=1` 和 batch 512；标准化仅用于监督目标。按验证 IC 选模；动态早停耐心为最佳 epoch 的 60% 向上取整、最少 4 轮、最多 12 轮，严格创新高即重置。
本运行从头使用全局随机样本和 FP16 输入缓存训练，没有迁移或重训其他模型权重。推理保留冻结配置指定的 FP16 舍入及 FP32 极值恢复，模型计算为 FP32；训练时 GRU 状态递推也保持 FP32。
`factor_scale=1`，直接输出原始模型分数，单位为标准化目标单位，未校准为预测收益。公共评估器对原始收益标签计算的 MSE／MAE 不代表标准化目标上的拟合误差；IC／Rank IC 仍使用官方原始标签。
三路窗口为 480／240／160 根，分别覆盖 2／5／20 个交易日。每路将 75 通道投影为 96 维，先用单层日内 GRU 编码，再用单层跨日 GRU 汇总。无 LayerNorm，总参数为 433,409；Dropout 为 `0.1`，推理时关闭。预测时钟决定 1 分钟分支的可见前缀，只读取最后已发生槽位及最后已开始交易日；09:30 读取昨日状态，未来补零不改变输出。
推理依赖：numpy>=1.26,<3, pandas>=2.2,<3, pyarrow>=14, torch>=2.6,<3，以及平台 `dai`。有 CUDA 时使用 GPU，否则使用 CPU；平台环境与成绩须实际验证。
训练复现：先安装 JSON 所附 `requirements-training.txt` 对应依赖，提供授权原始 Parquet，再显式调用 `reproduce("reproduced-v301", "/path/to/data", training_confirmed=True)`。
复现工作目录必须不存在；该调用会生成预处理数据并从头全局训练当前配置。硬件和软件环境差异可能影响重新训练的数值结果；冻结权重可直接用于推理。仅执行下方调试单元格不会训练。
本地评估摘要与原权重 SHA-256 见 JSON 的 `metadata`；逐文件哈希及本地入口验证记录位于仓库 `experiments/v301-global-fp16-s42-submission.json`，无需上传。


In [ ]:
"""两文件提交的 Notebook 入口模板；导入时不读取数据、不创建附件。"""

_RUNTIME = None


def _bundle_directory():
    from pathlib import Path
    return Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


def _read_bundle():
    import json
    raw = (_bundle_directory() / "v301-model.json").read_bytes()
    return json.loads(raw.decode("utf-8-sig"))


def _load_modules(bundle, paths):
    import torch
    import builtins
    from pathlib import Path
    from types import ModuleType
    modules = {}

    def local_import(name, globals=None, locals=None, fromlist=(), level=0):
        if level == 0 and name in modules:
            return modules[name]
        return builtins.__import__(name, globals, locals, fromlist, level)

    for relative in paths:
        entry = bundle["sources"][relative]
        name = Path(relative).stem
        module = ModuleType(name)
        module.__file__ = str(_bundle_directory() / Path(relative).name)
        module.__dict__["__builtins__"] = dict(vars(builtins), __import__=local_import)
        modules[name] = module
        exec(compile(entry["text"], module.__file__, "exec"), module.__dict__)
    return modules


def _runtime():
    global _RUNTIME
    if _RUNTIME is None:
        bundle = _read_bundle()
        modules = _load_modules(bundle, bundle["runtime_sources"])
        _RUNTIME = bundle, modules
    return _RUNTIME


def main(datasources, start_date, end_date):
    bundle, modules = _runtime()
    return modules["v301_predict"].predict(
        datasources, start_date, end_date, model_dir=_bundle_directory(), bundle=bundle,
    )


def reproduce(workspace, data_root, *, training_confirmed=False):
    """显式调用才启动预处理和训练；workspace 必须是不存在的新目录。"""
    if not training_confirmed:
        raise ValueError("v301 训练复现尚待用户明确确认")
    bundle = _read_bundle()
    modules = _load_modules(bundle, ["solutions/v301/v301_reproduce.py"])
    return modules["v301_reproduce"].reproduce(bundle, workspace, data_root,
                                           training_confirmed=True)


In [ ]:
if __name__ == "__main__":
    result = main(
        {f: "cpt_jyc_2026_stock_" + f for f in ["bar1m", "bar5m", "bar30m"]},
        "2024-12-30", "2024-12-31",
    )
    import numpy as np
    import pandas as pd
    assert list(result.columns) == ["date", "instrument", "factor"]
    assert not result.empty and result["factor"].dtype == np.dtype("float64")
    assert pd.api.types.is_datetime64_any_dtype(result["date"])
    assert result["instrument"].map(lambda value: isinstance(value, str)).all()
    assert not result[["date", "instrument"]].isna().any().any()
    assert not result.duplicated(["date", "instrument"]).any()
    assert np.isfinite(result["factor"]).all()
    print("输出行数：", len(result))
    print(result.dtypes)
    display(result.head())
